In [2]:
# =====================================================
# CROP RECOMMENDATION SYSTEM
# PROFESSIONAL ML PIPELINE
# =====================================================

# Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

# =====================================================
# 1. LOAD DATA
# =====================================================

df = pd.read_csv("Crop_recommendation.csv")

print("Shape:", df.shape)
print(df.head())

# =====================================================
# 2. BASIC DATA CHECKS
# =====================================================

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

# Remove duplicates if any
df = df.drop_duplicates()

# =====================================================
# 3. FEATURE & TARGET SEPARATION
# =====================================================

X = df.drop("label", axis=1)
y = df["label"]

# =====================================================
# 4. LABEL ENCODING
# =====================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# Save encoder
joblib.dump(label_encoder, "label_encoder.pkl")

# =====================================================
# 5. TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# =====================================================
# 6. PIPELINE
# =====================================================

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

# =====================================================
# 7. HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [10, 15, 20, None],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\nBest Parameters")
print(grid_search.best_params_)

print("\nBest CV Accuracy")
print(grid_search.best_score_)

# =====================================================
# 8. BEST MODEL
# =====================================================

best_model = grid_search.best_estimator_

# =====================================================
# 9. PREDICTION
# =====================================================

y_pred = best_model.predict(X_test)

# =====================================================
# 10. EVALUATION
# =====================================================

accuracy = accuracy_score(y_test, y_pred)

print("\nTest Accuracy")
print(accuracy)

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

# =====================================================
# 11. CROSS VALIDATION SCORE
# =====================================================

cv_scores = cross_val_score(
    best_model,
    X,
    y_encoded,
    cv=5,
    scoring='accuracy'
)

print("\nCross Validation Scores")
print(cv_scores)

print("\nAverage CV Accuracy")
print(cv_scores.mean())

# =====================================================
# 12. SAVE MODEL
# =====================================================

joblib.dump(best_model, "crop_recommendation_model.pkl")

print("\nModel Saved Successfully")

Shape: (2200, 8)
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice

Missing Values
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

Duplicates: 0
Train Shape: (1760, 7)
Test Shape : (440, 7)
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Parameters
{'classifier__max_depth': 15, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}

Best CV Accuracy
0.9954545454545455

Test Accuracy
0.9954545454545455

Classification Report
              precision    recall  f1-score   support



In [3]:
joblib.dump(
    best_model,
    r"C:\Users\chockalingam\Documents\Agriculture related Datasets\archive (13)\crop_recommendation_model.pkl"
)

joblib.dump(
    label_encoder,
    r"C:\Users\chockalingam\Documents\Agriculture related Datasets\archive (13)\label_encoder.pkl"
)

['C:\\Users\\chockalingam\\Documents\\Agriculture related Datasets\\archive (13)\\label_encoder.pkl']